In [30]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict,Annotated,Literal
from langgraph.graph import StateGraph, MessagesState, START, END
#this imports the checkpoiters and save the data into the RAM
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

True

In [31]:
llm = ChatGoogleGenerativeAI(model="gemini-3.7-flash")

Persistence

In [32]:
# main concept of persistance is like building things in checkpoints like whenever moving down in a graph we save the current state data and the older state data for our use till the point execution is done of the graph 

#we will assign a thread id for every execution if we want any partiuclar execution results we can just choose the thread id for that particular excution

In [33]:
class Jokestate(TypedDict):
    topic: str
    joke: str
    explanation: str

In [34]:
def generate_joke(state:Jokestate):
   prompt=f'Generate a joke on the topic->{state["topic"]}'
   result=llm.invoke(prompt).content
   
   return {'joke':result}

In [35]:
def generate_explanation(state:Jokestate):
    prompt = f'Generate an explanation of the joke: {state["joke"]}'
    result=llm.invoke(prompt).content
    
    return {'explanation':result}

In [36]:
graph = StateGraph(Jokestate)
graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [37]:
config1={"configurable":{"thread_id":"1"}}
workflow.invoke({'topic':'pizza'},config=config1)

GooglePermissionDeniedError: Error calling model 'gemini-3.7-flash' (PERMISSION_DENIED): 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your API key was reported as leaked. Please use another API key.', 'status': 'PERMISSION_DENIED'}}